# Clustering Algorithms for Beginners

## What is Clustering?

Clustering is an **unsupervised learning** technique that groups similar data points together WITHOUT labeled data.

## What You'll Learn

1. **K-Means Clustering** - Partitioning into k clusters
2. **Hierarchical Clustering** - Tree-based clustering
3. **DBSCAN** - Density-based clustering
4. **Gaussian Mixture Models** - Probabilistic clustering

**Real-World Applications:**
- Customer segmentation
- Image compression
- Anomaly detection
- Document grouping
- Social network analysis

---

## Setup: Import Libraries

In [ ]:
# Install required packages (uncomment if needed)
# !pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA

# For hierarchical clustering visualization
from scipy.cluster.hierarchy import dendrogram, linkage

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## Understanding Clustering

### Supervised vs Unsupervised Learning

| Aspect | Supervised | Unsupervised (Clustering) |
|--------|------------|---------------------------|
| Labels | Has labels | No labels |
| Goal | Predict known categories | Discover hidden groups |
| Evaluation | Accuracy, F1, etc. | Silhouette, Inertia |
| Example | Spam detection | Customer segmentation |

## Create Sample Data for Visualization

In [ ]:
from sklearn.datasets import make_blobs, make_moons, make_circles

# Generate different types of data
np.random.seed(42)

# Blob data (good for K-Means)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Moon data (good for DBSCAN)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

# Circle data (challenging for K-Means)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', s=30)
axes[0].set_title('Blob Clusters\n(Good for K-Means)')

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=30)
axes[1].set_title('Moon Clusters\n(Good for DBSCAN)')

axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=30)
axes[2].set_title('Circle Clusters\n(Challenging for K-Means)')

plt.tight_layout()
plt.show()

## Load Real Dataset: Mall Customer Segmentation

This is a classic dataset for practicing customer segmentation.

In [ ]:
# Load Mall Customer dataset
# Dataset available from Kaggle: https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python

# For this tutorial, we'll create a similar dataset
np.random.seed(42)

n_customers = 200

# Create customer segments
# Segment 1: High income, high spending
seg1_income = np.random.normal(80, 10, 40)
seg1_spending = np.random.normal(80, 10, 40)

# Segment 2: High income, low spending (savers)
seg2_income = np.random.normal(80, 10, 40)
seg2_spending = np.random.normal(20, 10, 40)

# Segment 3: Low income, high spending (spenders)
seg3_income = np.random.normal(20, 10, 40)
seg3_spending = np.random.normal(80, 10, 40)

# Segment 4: Low income, low spending
seg4_income = np.random.normal(20, 10, 40)
seg4_spending = np.random.normal(20, 10, 40)

# Segment 5: Average
seg5_income = np.random.normal(50, 15, 40)
seg5_spending = np.random.normal(50, 15, 40)

# Combine
annual_income = np.concatenate([seg1_income, seg2_income, seg3_income, seg4_income, seg5_income])
spending_score = np.concatenate([seg1_spending, seg2_spending, seg3_spending, seg4_spending, seg5_spending])

# Clip to valid range
annual_income = np.clip(annual_income, 1, 100)
spending_score = np.clip(spending_score, 1, 100)

# Create DataFrame
customers_df = pd.DataFrame({
    'CustomerID': range(1, n_customers + 1),
    'Gender': np.random.choice(['Male', 'Female'], n_customers),
    'Age': np.random.randint(18, 70, n_customers),
    'Annual_Income': annual_income.astype(int),
    'Spending_Score': spending_score.astype(int)
})

print("Mall Customer Dataset")
print(f"Shape: {customers_df.shape}")
customers_df.head(10)

## Exploratory Data Analysis

In [ ]:
# Statistical summary
print("Statistical Summary:")
customers_df.describe()

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
axes[0].hist(customers_df['Age'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Age Distribution')

# Income distribution
axes[1].hist(customers_df['Annual_Income'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Annual Income (k$)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Annual Income Distribution')

# Spending Score distribution
axes[2].hist(customers_df['Spending_Score'], bins=20, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_xlabel('Spending Score (1-100)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Spending Score Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Income vs Spending Score scatter plot
plt.figure(figsize=(10, 8))
plt.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'], 
            c='steelblue', s=50, alpha=0.7)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Distribution: Income vs Spending\n(Can you see potential clusters?)')
plt.grid(True, alpha=0.3)
plt.show()

## Clustering Evaluation Metrics

Since we don't have labels, we use internal metrics:

| Metric | What it Measures | Good Values |
|--------|------------------|-------------|
| **Silhouette Score** | How similar points are to their cluster vs other clusters | Close to 1 |
| **Inertia** | Sum of squared distances to cluster centers | Lower is better |
| **Calinski-Harabasz** | Ratio of between-cluster to within-cluster variance | Higher is better |
| **Davies-Bouldin** | Average similarity between clusters | Lower is better |

In [ ]:
# Helper function to evaluate clustering
def evaluate_clustering(X, labels, algorithm_name):
    """
    Evaluate clustering using multiple metrics
    """
    # Filter out noise points for DBSCAN (label = -1)
    mask = labels != -1
    if mask.sum() < 2:
        return {'Algorithm': algorithm_name, 'Silhouette': np.nan, 
                'Calinski-Harabasz': np.nan, 'Davies-Bouldin': np.nan}
    
    n_clusters = len(set(labels[mask]))
    if n_clusters < 2:
        return {'Algorithm': algorithm_name, 'Silhouette': np.nan, 
                'Calinski-Harabasz': np.nan, 'Davies-Bouldin': np.nan}
    
    results = {
        'Algorithm': algorithm_name,
        'N Clusters': n_clusters,
        'Silhouette': silhouette_score(X[mask], labels[mask]),
        'Calinski-Harabasz': calinski_harabasz_score(X[mask], labels[mask]),
        'Davies-Bouldin': davies_bouldin_score(X[mask], labels[mask])
    }
    
    return results

## Data Preparation

In [ ]:
# Select features for clustering
# We'll use Annual Income and Spending Score
X = customers_df[['Annual_Income', 'Spending_Score']].values

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Data shape: {X_scaled.shape}")
print("Features scaled for clustering")

---

# Algorithm 1: K-Means Clustering

## The Most Popular Clustering Algorithm

**Concept:** Partition data into k clusters by minimizing within-cluster variance.

**How it works:**
1. Initialize k cluster centers (centroids) randomly
2. Assign each point to nearest centroid
3. Update centroids as mean of assigned points
4. Repeat steps 2-3 until convergence

**Key Parameter:** k (number of clusters) - must be specified!

In [ ]:
# Visualize K-Means algorithm steps
from sklearn.cluster import KMeans

# Simple visualization data
X_demo = X_scaled[:100]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Step 1: Initial random centroids
np.random.seed(42)
initial_centroids = X_demo[np.random.choice(len(X_demo), 3, replace=False)]
axes[0].scatter(X_demo[:, 0], X_demo[:, 1], c='gray', s=30)
axes[0].scatter(initial_centroids[:, 0], initial_centroids[:, 1], c='red', s=200, marker='X')
axes[0].set_title('Step 1: Random Centroids')

# Steps 2-4: K-Means iterations
for i, n_iter in enumerate([1, 2, 10], start=1):
    kmeans = KMeans(n_clusters=3, init=initial_centroids, n_init=1, max_iter=n_iter, random_state=42)
    labels = kmeans.fit_predict(X_demo)
    
    axes[i].scatter(X_demo[:, 0], X_demo[:, 1], c=labels, cmap='viridis', s=30)
    axes[i].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
                   c='red', s=200, marker='X')
    axes[i].set_title(f'Step {i+1}: After {n_iter} iteration(s)')

plt.tight_layout()
plt.show()

### Finding Optimal k: Elbow Method and Silhouette Analysis

In [ ]:
# Elbow Method
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)')
axes[0].set_title('Elbow Method\n(Look for the "elbow" point)')
axes[0].grid(True, alpha=0.3)

# Silhouette plot
axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis\n(Higher is better)')
axes[1].grid(True, alpha=0.3)

# Mark optimal k
optimal_k = k_range[np.argmax(silhouette_scores)]
axes[1].axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k={optimal_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Optimal k based on Silhouette Score: {optimal_k}")

In [ ]:
# Final K-Means clustering
kmeans_model = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans_model.fit_predict(X_scaled)

# Evaluate
kmeans_results = evaluate_clustering(X_scaled, kmeans_labels, 'K-Means')

print("=" * 50)
print("K-MEANS CLUSTERING RESULTS")
print("=" * 50)
for key, value in kmeans_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize K-Means clusters
plt.figure(figsize=(12, 8))

# Plot clusters
scatter = plt.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'],
                      c=kmeans_labels, cmap='viridis', s=60, alpha=0.7)

# Plot centroids (transform back to original scale)
centroids_original = scaler.inverse_transform(kmeans_model.cluster_centers_)
plt.scatter(centroids_original[:, 0], centroids_original[:, 1],
            c='red', s=300, marker='X', edgecolors='black', linewidths=2, label='Centroids')

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('K-Means Customer Segmentation (k=5)')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Analyze cluster characteristics
customers_df['Cluster'] = kmeans_labels

cluster_summary = customers_df.groupby('Cluster').agg({
    'Annual_Income': ['mean', 'min', 'max'],
    'Spending_Score': ['mean', 'min', 'max'],
    'Age': 'mean',
    'CustomerID': 'count'
}).round(1)

print("\nCluster Summary:")
print(cluster_summary)

In [ ]:
# Name the clusters based on characteristics
cluster_names = {
    0: 'Low Income, Low Spending',
    1: 'High Income, Low Spending (Savers)',
    2: 'Average Customers',
    3: 'Low Income, High Spending',
    4: 'High Income, High Spending (Target)'
}

# Visualize with names
plt.figure(figsize=(12, 8))

colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
for cluster_id in range(5):
    mask = kmeans_labels == cluster_id
    plt.scatter(customers_df.loc[mask, 'Annual_Income'], 
                customers_df.loc[mask, 'Spending_Score'],
                c=colors[cluster_id], s=60, alpha=0.7, 
                label=cluster_names.get(cluster_id, f'Cluster {cluster_id}'))

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Customer Segments Identified by K-Means')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Algorithm 2: Hierarchical Clustering

## Building a Cluster Tree

**Concept:** Build a hierarchy of clusters, can be visualized as a tree (dendrogram).

**Two Approaches:**
- **Agglomerative (bottom-up):** Start with each point as a cluster, merge similar clusters
- **Divisive (top-down):** Start with one cluster, recursively split

**Linkage Methods:**
- **Single:** Distance between closest points
- **Complete:** Distance between farthest points
- **Average:** Average distance between all pairs
- **Ward:** Minimizes within-cluster variance

In [ ]:
# Create dendrogram
plt.figure(figsize=(14, 8))

# Calculate linkage
linkage_matrix = linkage(X_scaled, method='ward')

# Plot dendrogram
dendrogram(linkage_matrix, 
           truncate_mode='level', 
           p=5,  # Show only last p merges
           leaf_rotation=90,
           leaf_font_size=10,
           color_threshold=7)

plt.xlabel('Data Points')
plt.ylabel('Distance')
plt.title('Dendrogram: Hierarchical Clustering\n(Cut at different heights for different numbers of clusters)')
plt.axhline(y=7, color='red', linestyle='--', label='Cut for 5 clusters')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative Clustering
hierarchical_model = AgglomerativeClustering(n_clusters=5, linkage='ward')
hierarchical_labels = hierarchical_model.fit_predict(X_scaled)

# Evaluate
hierarchical_results = evaluate_clustering(X_scaled, hierarchical_labels, 'Hierarchical')

print("=" * 50)
print("HIERARCHICAL CLUSTERING RESULTS")
print("=" * 50)
for key, value in hierarchical_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize Hierarchical clusters
plt.figure(figsize=(12, 8))

scatter = plt.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'],
                      c=hierarchical_labels, cmap='viridis', s=60, alpha=0.7)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Hierarchical Clustering Results (Ward Linkage)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

---

# Algorithm 3: DBSCAN

## Density-Based Clustering

**Concept:** Form clusters based on density of points. Points in low-density regions are outliers.

**How it works:**
1. For each point, find neighbors within distance ε (epsilon)
2. If ≥ min_samples neighbors → core point
3. Connect core points and their neighbors into clusters
4. Points not belonging to any cluster = noise (outliers)

**Key Parameters:**
- **eps (ε):** Maximum distance between neighbors
- **min_samples:** Minimum points to form a cluster

**Advantages:**
- No need to specify number of clusters
- Finds arbitrary-shaped clusters
- Identifies outliers

In [ ]:
# DBSCAN works great on moon-shaped data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# K-Means on moons (fails)
kmeans_moons = KMeans(n_clusters=2, random_state=42)
labels_km_moons = kmeans_moons.fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km_moons, cmap='viridis', s=30)
axes[0].set_title('K-Means on Moons\n(Fails to capture shape)')

# DBSCAN on moons (succeeds)
dbscan_moons = DBSCAN(eps=0.2, min_samples=5)
labels_db_moons = dbscan_moons.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db_moons, cmap='viridis', s=30)
axes[1].set_title('DBSCAN on Moons\n(Captures shape correctly!)')

# DBSCAN on circles
dbscan_circles = DBSCAN(eps=0.2, min_samples=5)
labels_db_circles = dbscan_circles.fit_predict(X_circles)
axes[2].scatter(X_circles[:, 0], X_circles[:, 1], c=labels_db_circles, cmap='viridis', s=30)
axes[2].set_title('DBSCAN on Circles\n(Handles concentric circles)')

plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN on customer data
# Finding optimal eps using k-distance plot
from sklearn.neighbors import NearestNeighbors

# Calculate distances to 4th nearest neighbor
neighbors = NearestNeighbors(n_neighbors=5)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)

# Sort distances
k_distances = np.sort(distances[:, 4])[::-1]

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.xlabel('Points (sorted by distance)')
plt.ylabel('Distance to 4th Nearest Neighbor')
plt.title('K-Distance Plot\n(Look for "elbow" to choose eps)')
plt.axhline(y=0.5, color='red', linestyle='--', label='eps = 0.5')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# DBSCAN clustering
dbscan_model = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan_model.fit_predict(X_scaled)

# Count clusters and noise
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"Number of clusters: {n_clusters}")
print(f"Number of noise points: {n_noise}")

# Evaluate
dbscan_results = evaluate_clustering(X_scaled, dbscan_labels, 'DBSCAN')

print("\n" + "=" * 50)
print("DBSCAN CLUSTERING RESULTS")
print("=" * 50)
for key, value in dbscan_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize DBSCAN clusters
plt.figure(figsize=(12, 8))

# Plot clusters
unique_labels = set(dbscan_labels)
colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        # Noise points
        color = 'gray'
        marker = 'x'
        label_name = 'Noise (Outliers)'
    else:
        marker = 'o'
        label_name = f'Cluster {label}'
    
    mask = dbscan_labels == label
    plt.scatter(customers_df.loc[mask, 'Annual_Income'],
                customers_df.loc[mask, 'Spending_Score'],
                c=[color], s=60 if label != -1 else 30, marker=marker,
                alpha=0.7, label=label_name)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('DBSCAN Clustering Results\n(Gray X marks are outliers)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

# Algorithm 4: Gaussian Mixture Models (GMM)

## Probabilistic Clustering

**Concept:** Assume data is generated from a mixture of Gaussian distributions.

**How it works:**
1. Assume data comes from k Gaussian distributions
2. Estimate parameters (mean, covariance) for each Gaussian
3. Calculate probability of each point belonging to each cluster
4. Use soft assignments (probability) instead of hard assignments

**Advantages:**
- Provides probability of cluster membership
- Can model elliptical clusters
- More flexible than K-Means

In [ ]:
# GMM clustering
gmm_model = GaussianMixture(n_components=5, covariance_type='full', random_state=42)
gmm_labels = gmm_model.fit_predict(X_scaled)

# Get probabilities
gmm_probs = gmm_model.predict_proba(X_scaled)

# Evaluate
gmm_results = evaluate_clustering(X_scaled, gmm_labels, 'GMM')

print("=" * 50)
print("GAUSSIAN MIXTURE MODEL RESULTS")
print("=" * 50)
for key, value in gmm_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize GMM clusters
plt.figure(figsize=(12, 8))

scatter = plt.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'],
                      c=gmm_labels, cmap='viridis', s=60, alpha=0.7)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Gaussian Mixture Model Clustering')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Show probability distributions
# Find points with uncertain cluster membership
max_probs = gmm_probs.max(axis=1)
uncertain_mask = max_probs < 0.7  # Less than 70% confident

print(f"Points with uncertain cluster membership (prob < 70%): {uncertain_mask.sum()}")

plt.figure(figsize=(12, 8))

# Color by confidence
scatter = plt.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'],
                      c=max_probs, cmap='RdYlGn', s=60, alpha=0.7)

plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('GMM: Cluster Assignment Confidence\n(Green = High confidence, Red = Low confidence)')
plt.colorbar(scatter, label='Confidence')
plt.grid(True, alpha=0.3)
plt.show()

---

# Algorithm Comparison Summary

In [ ]:
# Compile all results
all_results = pd.DataFrame([
    kmeans_results,
    hierarchical_results,
    dbscan_results,
    gmm_results
])

print("=" * 70)
print("CLUSTERING ALGORITHM COMPARISON")
print("=" * 70)
print(all_results.round(4).to_string(index=False))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

algorithms = [
    ('K-Means', kmeans_labels),
    ('Hierarchical', hierarchical_labels),
    ('DBSCAN', dbscan_labels),
    ('GMM', gmm_labels)
]

for (name, labels), ax in zip(algorithms, axes.flatten()):
    scatter = ax.scatter(customers_df['Annual_Income'], customers_df['Spending_Score'],
                         c=labels, cmap='viridis', s=40, alpha=0.7)
    ax.set_xlabel('Annual Income (k$)')
    ax.set_ylabel('Spending Score')
    ax.set_title(f'{name} Clustering')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Key Takeaways

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| **K-Means** | Spherical clusters, large data | Fast, scalable | Need to specify k, sensitive to outliers |
| **Hierarchical** | Understanding hierarchy, small data | Dendrogram visualization, no k needed | Slow on large data |
| **DBSCAN** | Arbitrary shapes, outlier detection | No k needed, finds outliers | Sensitive to eps and min_samples |
| **GMM** | Probabilistic assignments, elliptical | Soft clustering, flexible shapes | Computationally expensive |

---

## Practice Exercises

1. Try clustering with more features (Age + Income + Spending)
2. Use PCA to visualize high-dimensional clusters
3. Apply clustering to the Iris dataset (compare with true labels)
4. Experiment with different DBSCAN parameters

In [ ]:
# Exercise: Clustering with 3 features
X_3d = customers_df[['Age', 'Annual_Income', 'Spending_Score']].values
X_3d_scaled = StandardScaler().fit_transform(X_3d)

# Your code here:
# 1. Apply K-Means with k=5
# 2. Use PCA to reduce to 2D for visualization
# 3. Plot the clusters